# 图像基本运算


## 1. 灰度化 (Grayscale Conversion)


灰度化是将彩色图像转换为灰度图像的过程。彩色图像通常包含红、绿、蓝三个颜色通道，而灰度图像只有一个亮度通道。灰度图像的每个像素值表示该点的亮度信息，通常在0（黑色）到255（白色）之间。

**为什么进行灰度化？**

*   **简化处理：** 灰度图像数据量更小，处理速度更快，许多算法（如边缘检测）在灰度图像上效果更好。
*   **突出结构：** 有时颜色信息会干扰对图像结构或纹理的分析。
*   **节省存储：** 减少图像文件大小。

**常见灰度化方法：**

最常用的方法是加权平均法，即根据人眼对不同颜色的敏感度，给RGB通道分配不同的权重：
灰度值 = 0.299 \* R + 0.587 \* G + 0.114 \* B

**实际应用场景：**

*   **人脸识别：** 很多早期的人脸识别算法在灰度图像上进行特征提取。
*   **OCR (光学字符识别)：** 识别文本时，通常先将图像灰度化或二值化，以提高字符识别的准确性。
*   **工业检测：** 在检测产品缺陷时，灰度图像可以更好地突出纹理和阴影变化。



In [ ]:
import cv2
import matplotlib.pyplot as plt

# 读取图像
image = cv2.imread('1.jpg')  # 读取为BGR格式

# 方法1：使用cvtColor函数（最常用）
gray_cv2 = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


plt.imshow(gray_cv2, cmap='gray')
plt.title('OpenCV灰度化')
plt.axis('off')

## 2. 二值化 (Binarization)

二值化是图像处理中一种将灰度图像转换为只包含两种颜色（通常是黑色和白色）的图像的方法。它通过设定一个**阈值**，将图像中像素值高于阈值的点设为一种颜色（如白色），低于或等于阈值的点设为另一种颜色（如黑色）。

**为什么进行二值化？**

*   **进一步简化：** 将图像信息简化到极致，只保留最重要的特征（如前景与背景）。
*   **突出轮廓：** 有助于分离图像中的目标物体与背景。
*   **提高效率：** 减少后续处理的数据量和计算复杂度。

**阈值选择：**

阈值的选择是二值化的关键。一个好的阈值能够有效地区分前景和背景。常见的阈值方法包括：

*   **固定阈值法：** 手动设定一个固定值。
*   **自适应阈值法：** 根据图像局部区域的特性自动计算阈值，适用于光照不均的图像。
*   **Otsu's 法：** 自动寻找最佳全局阈值，使得前景和背景的类间方差最大。

**实际应用场景：**

*   **文档扫描与识别：** 将扫描的文档二值化，可以清晰地分离文字和纸张背景，便于OCR识别。
*   **二维码识别：** 二维码通常是黑白模式，二值化有助于准确识别其图案。
*   **缺陷检测：** 在工业生产中，二值化可以帮助快速识别产品表面的划痕、斑点等缺陷。



下面这段代码我们展示三种二值化的方法。

| 特性 | 固定阈值二值化 | 自适应阈值二值化 | Otsu二值化 |
|------|---------------|-----------------|-----------|
| **原理** | 使用固定的全局阈值 | 根据局部区域像素计算阈值 | 基于图像直方图自动确定最佳阈值 |
| **阈值确定** | 手动设置（如127） | 自动计算局部阈值 | 自动计算全局最优阈值 |
| **适用场景** | 光照均匀的图像 | 光照不均匀的图像 | 双峰直方图的图像 |
| **计算复杂度** | 低 | 中 | 中 |
| **参数设置** | 需要手动调整阈值 | 需要设置邻域大小和常数 | 无需参数，自动计算 |
| **优点** | 计算简单快速 | 适应光照变化 | 自动确定最佳阈值 |
| **缺点** | 对光照敏感 | 计算量较大 | 对非双峰直方图效果差 |
| **OpenCV函数** | `cv2.threshold(THRESH_BINARY)` | `cv2.adaptiveThreshold()` | `cv2.threshold(THRESH_OTSU)` |


In [ ]:
# 方法1：固定阈值法
def simple_threshold(image, threshold=127):
    # 将大于阈值的像素设为255，小于等于的设为0
    return (image > threshold) * 255

binary_image = simple_threshold(gray_cv2, 127)

# 方法2：使用OpenCV的阈值函数
_, thresh_binary = cv2.threshold(gray_cv2, 127, 255, cv2.THRESH_BINARY)

# 方法3：自适应阈值法（适用于光照不均匀的图像）
thresh_adaptive = cv2.adaptiveThreshold(
    gray_cv2, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 11, 2
)

# 方法4：Otsu's二值化（自动确定最佳阈值）
_, thresh_otsu = cv2.threshold(
    gray_cv2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

# 显示二值化结果
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(thresh_binary, cmap='gray')
plt.title('固定阈值二值化')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(thresh_adaptive, cmap='gray')
plt.title('自适应阈值二值化')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(thresh_otsu, cmap='gray')
plt.title('Otsu二值化')
plt.axis('off')

plt.tight_layout()
plt.show()

## 3. 亮度与对比度调整 (Brightness & Contrast Adjustment)



亮度与对比度是图像质量的两个重要指标。

*   **亮度 (Brightness)：** 指图像的明暗程度。增加亮度会使图像整体变亮，减少亮度会使图像整体变暗。
*   **对比度 (Contrast)：** 指图像中最亮和最暗区域之间的差异程度。增加对比度会使图像的亮部更亮、暗部更暗，从而使图像看起来更鲜明、有层次感；降低对比度则会使图像变“灰”，细节模糊。

**调整原理：**

图像的亮度与对比度调整可以通过对每个像素值进行线性变换来实现。
新的像素值 = `alpha` \* 原始像素值 + `beta`

*   `alpha` (通常大于0)：控制对比度。`alpha > 1` 增加对比度，`0 < alpha < 1` 降低对比度。
*   `beta` (通常在-255到255之间)：控制亮度。`beta > 0` 增加亮度，`beta < 0` 降低亮度。

**实际应用场景：**

*   **照片后期处理：** 调整照片的明暗和层次感，使其更具表现力。
*   **医学影像增强：** 提高病灶区域的对比度，帮助医生更清晰地观察。
*   **监控视频优化：** 在光线不足或过曝的环境下，调整亮度和对比度以看清细节。

**代码示例：调整图像亮度与对比度**




In [ ]:
import cv2
import matplotlib.pyplot as plt

# 尝试加载实际图片
try:
    original_img = cv2.imread('1.jpg')  # 默认以彩色模式加载
    if original_img is None:
        raise FileNotFoundError("图片未找到，请确保图片路径正确。")
except Exception as e:
    print(f"错误: {e}")
    exit()

# 调整亮度 (beta = 50)
beta_val = 50
bright_img = cv2.add(original_img, (beta_val, beta_val, beta_val, 0))  # 只对BGR通道加值

# 调整对比度 (alpha = 1.5)
alpha_val = 1.5
contrast_img = cv2.convertScaleAbs(original_img, alpha=alpha_val, beta=0)

# 同时调整亮度和对比度
alpha_both = 1.5
beta_both = 30
adjusted_img = cv2.convertScaleAbs(original_img, alpha=alpha_both, beta=beta_both)

# 显示所有图像
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.imshow(cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB))
plt.title('原始图像')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(cv2.cvtColor(bright_img, cv2.COLOR_BGR2RGB))
plt.title(f'亮度增加 (beta={beta_val})')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(cv2.cvtColor(contrast_img, cv2.COLOR_BGR2RGB))
plt.title(f'对比度增加 (alpha={alpha_val})')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(cv2.cvtColor(adjusted_img, cv2.COLOR_BGR2RGB))
plt.title(f'亮度对比度同时调整 (alpha={alpha_both}, beta={beta_both})')
plt.axis('off')

plt.show()

## 4. 图像的加法与减法 (Image Addition & Subtraction)


图像的加法和减法是对两幅或多幅图像进行像素级别的运算。

*   **图像加法：** 将两幅图像对应位置的像素值相加。这可以用于图像融合、叠加水印、去除噪声（通过叠加多张相同场景的噪声图像来平均噪声）。
*   **图像减法：** 将一幅图像的像素值减去另一幅图像对应位置的像素值。这可以用于背景消除、运动检测、变化检测。

**注意事项：**

*   **图像尺寸必须相同：** 进行加减运算的两幅图像必须有相同的尺寸（宽度、高度和通道数）。
*   **像素值溢出：** 像素值相加可能超过255，相减可能低于0。OpenCV的 `cv2.add()` 和 `cv2.subtract()` 会自动处理溢出（饱和运算），而NumPy的直接加减会进行模运算。通常使用OpenCV的函数更符合图像处理的直观效果。

**实际应用场景：**

*   **图像加法：**
    *   **图像融合：** 将不同曝光的图像叠加，得到细节更丰富的HDR图像。
    *   **叠加水印/Logo：** 将半透明的Logo叠加到图像上。
    *   **去噪：** 对多张相同场景的图像求平均，可以有效抑制随机噪声。
*   **图像减法：**
    *   **背景消除：** 从包含目标的图像中减去纯背景图像，得到目标本身。
    *   **运动检测：** 连续帧图像相减，可以突出显示运动的物体。
    *   **变化检测：** 对比同一场景不同时间拍摄的两张图像，找出变化区域。

**代码示例：图像加法与减法**





In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 创建两张简单的图像用于演示
# 图像1: 红色矩形
img1 = np.zeros((100, 100, 3), dtype=np.uint8)
cv2.rectangle(img1, (10, 10), (90, 90), (0, 0, 255), -1)  # 红色矩形

# 图像2: 蓝色矩形，并与图像1有部分重叠
img2 = np.zeros((100, 100, 3), dtype=np.uint8)
cv2.rectangle(img2, (30, 30), (70, 70), (255, 0, 0), -1)  # 蓝色矩形

# 图像加法 (使用cv2.add，会进行饱和运算)
# 红色 + 蓝色 = 紫色 (在重叠区域)
added_img = cv2.add(img1, img2)

# 图像减法 (使用cv2.subtract，会进行饱和运算)
# 从红色图像中减去蓝色图像
subtracted_img = cv2.subtract(img1, img2)

# 显示所有图像并排
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
plt.title('图像1 (红色)')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
plt.title('图像2 (蓝色)')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(cv2.cvtColor(added_img, cv2.COLOR_BGR2RGB))
plt.title('图像加法 (img1 + img2)')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(cv2.cvtColor(subtracted_img, cv2.COLOR_BGR2RGB))
plt.title('图像减法 (img1 - img2)')
plt.axis('off')

plt.show()



**预期结果:**
1. 图像1 (红色)：显示一个纯红色矩形。
2. 图像2 (蓝色)：显示一个中间是纯蓝色的矩形。
3. 图像加法 (img1 + img2)：
   - 非重叠红色区域保持红色。
   - 非重叠蓝色区域保持蓝色。
   - 重叠区域 (红色 + 蓝色) 变为紫色 (因为红色和蓝色通道都达到 255)。
4. 图像减法 (img1 - img2)：
   - 非重叠红色区域保持红色。
   - 非重叠蓝色区域变为黑色 (因为原始图像1在该区域为 0，减去图像2的蓝色255，结果饱和为 0)。
   - 重叠区域 (红色 - 蓝色) 红色通道不变，蓝色通道变为 0，因此保持红色。

**代码解释：**

1.  `cv2.add(img1, img2)`: 对两幅图像进行像素级别的加法运算。当结果超过255时，会自动截断为255。
2.  `cv2.subtract(img1, img2)`: 对两幅图像进行像素级别的减法运算。当结果低于0时，会自动截断为0。
